#Stage: Import packages

In [1]:
import sys
import os

print (os.getcwd())

/Users/niloy/Library/CloudStorage/OneDrive-Personal/Desktop/My Projects/Mutual Fund Analyzer/emfer_code/notebooks


In [2]:
os.chdir("..")
print (os.getcwd())

/Users/niloy/Library/CloudStorage/OneDrive-Personal/Desktop/My Projects/Mutual Fund Analyzer/emfer_code


#Stage: Import MF NAV history

In [3]:
from src.emfer.data.mf_api import fetch_nav_history
mf_df, mf_name = fetch_nav_history(125497)



/Users/niloy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
print (mf_name)
mf_df.head()


SBI Small Cap Fund - Direct Plan - Growth


,date,nav,fund_name
3086,2013-11-18,13.0894,SBI Small Cap Fund - Direct Plan - Growth
3085,2013-11-19,13.1068,SBI Small Cap Fund - Direct Plan - Growth
3084,2013-11-20,12.9549,SBI Small Cap Fund - Direct Plan - Growth
3083,2013-11-21,12.7957,SBI Small Cap Fund - Direct Plan - Growth
3082,2013-11-22,12.7775,SBI Small Cap Fund - Direct Plan - Growth


In [5]:
from src.emfer.data.rolling_returns import calculate_rolling_returns, get_nearest_past_index
df_rolling = calculate_rolling_returns(mf_df, n=5)
#df_rolling.head()

In [6]:
df_rolling.head()

,current_date,past_date,cagr_5_years,current_nav,past_nav,fund_name,fund
0,2018-11-19,2013-11-19,32.252336,53.0290,13.1068,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
1,2018-11-20,2013-11-20,32.402064,52.7118,12.9549,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
2,2018-11-21,2013-11-21,32.714131,52.6805,12.7957,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
3,2018-11-22,2013-11-22,32.704609,52.5867,12.7775,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
4,2018-11-26,2013-11-26,32.425943,52.3692,12.8591,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund


In [9]:
from src.emfer.charts.charts import plot_nav
plot_nav(df_rolling)


KeyError: 'date'

In [10]:
from src.emfer.charts.charts import rolling_returns_summary
df_summary = rolling_returns_summary(df_rolling, 5)


In [11]:
df_summary.head()

,fund_name,start_date,end_date,metric,latest_5_year_cagr,mean,std_dev,min,p5,p10,p25,median,p75,p90,p95,max
0,SBI Small Cap Fund - Direct Plan - Growth,2013-11-19,2026-05-26,5Y CAGR,14.82,21.81,5.95,7.4,11.62,14.49,17.19,21.91,25.95,30.31,31.34,34.51


In [12]:
from src.emfer.genAI.fund_store import create_fund_store
fund_store = create_fund_store(df_rolling, df_summary)

In [13]:
fund_store.keys()

dict_keys(['SBI Small Cap Fund - Direct Plan - Growth'])

In [14]:
fund_store['SBI Small Cap Fund - Direct Plan - Growth'].keys()

dict_keys(['rolling_nav_cagr', 'summary'])

In [15]:
fund_store['SBI Small Cap Fund - Direct Plan - Growth']['rolling_nav_cagr'].head()

,current_date,past_date,cagr_5_years,current_nav,past_nav,fund_name,fund
0,2018-11-19,2013-11-19,32.252336,53.0290,13.1068,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
1,2018-11-20,2013-11-20,32.402064,52.7118,12.9549,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
2,2018-11-21,2013-11-21,32.714131,52.6805,12.7957,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
3,2018-11-22,2013-11-22,32.704609,52.5867,12.7775,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund
4,2018-11-26,2013-11-26,32.425943,52.3692,12.8591,SBI Small Cap Fund - Direct Plan - Growth,SBI Small Cap Fund


In [16]:
from src.emfer.genAI.fund_store import build_rag_context
rag_context = build_rag_context(fund_store)

In [17]:
print (rag_context)


                            Fund Name: SBI Small Cap Fund - Direct Plan - Growth
                            
                            Summary:
                                                            fund_name start_date   end_date  metric  latest_5_year_cagr  mean  std_dev  min    p5   p10   p25  median   p75   p90   p95   max
SBI Small Cap Fund - Direct Plan - Growth 2013-11-19 2026-05-26 5Y CAGR               14.82 21.81     5.95  7.4 11.62 14.49 17.19   21.91 25.95 30.31 31.34 34.51

                            Rolling NAV and CAGR Analysis:
                            current_date  past_date  cagr_5_years  current_nav  past_nav                                 fund_name               fund
  2018-11-19 2013-11-19     32.252336      53.0290   13.1068 SBI Small Cap Fund - Direct Plan - Growth SBI Small Cap Fund
  2018-11-20 2013-11-20     32.402064      52.7118   12.9549 SBI Small Cap Fund - Direct Plan - Growth SBI Small Cap Fund
  2018-11-21 2013-11-21     32.714131      52.

In [18]:
from src.emfer.genAI.prompts import system_instruction
sys_instruction = system_instruction(rag_context)

In [19]:
print (sys_instruction)


  You are Scout, an AI assistant for mutual fund analysis.

  Your role is to interpret mutual fund performance data in a clear, concise, intuitive, and investor-friendly way.

  Mutual Fund Summary Data:
  
                            Fund Name: SBI Small Cap Fund - Direct Plan - Growth
                            
                            Summary:
                                                            fund_name start_date   end_date  metric  latest_5_year_cagr  mean  std_dev  min    p5   p10   p25  median   p75   p90   p95   max
SBI Small Cap Fund - Direct Plan - Growth 2013-11-19 2026-05-26 5Y CAGR               14.82 21.81     5.95  7.4 11.62 14.49 17.19   21.91 25.95 30.31 31.34 34.51

                            Rolling NAV and CAGR Analysis:
                            current_date  past_date  cagr_5_years  current_nav  past_nav                                 fund_name               fund
  2018-11-19 2013-11-19     32.252336      53.0290   13.1068 SBI Small Cap Fund - 

In [22]:
from src.emfer.genAI.scout import ask_scout
model = ask_scout(sys_instruction)

In [23]:
from src.emfer.genAI.scout import scout_answer
answer = scout_answer(model, "Which funds do you have information on?")
print (answer)

I have information on the **SBI Small Cap Fund - Direct Plan - Growth**.
